# Attention And `torch.compile` Benchmarks

This notebook reuses the repo benchmark script, preferring `cs336_systems/implementations/benchmarking.py` and falling back to the legacy path when needed. It only changes the benchmark grids/configuration to match the handout requirements and produces three tables:

1. `pytorch_attention` timings and pre-backward memory
2. compiled vs uncompiled attention
3. vanilla vs compiled Transformer timings


In [11]:
from __future__ import annotations

import os
import subprocess
from pathlib import Path

REPO_URL = os.environ.get("ASSIGNMENT2_REPO_URL", "https://github.com/Eden-kk/assignment2-systems.git")
REPO_ROOT = Path("/content/assignment2-systems")

if REPO_ROOT.exists():
    print(f"Refreshing existing repo at {REPO_ROOT}")
    subprocess.run(["git", "fetch", "origin"], cwd=REPO_ROOT, check=True, timeout=300)
    subprocess.run(["git", "checkout", "main"], cwd=REPO_ROOT, check=True, timeout=300)
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO_ROOT, check=True, timeout=300)
else:
    print(f"Cloning {REPO_URL} into {REPO_ROOT} ...")
    subprocess.run(["git", "clone", "--depth", "1", "--progress", REPO_URL, str(REPO_ROOT)], check=True, timeout=300)

os.chdir(REPO_ROOT)
head = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
print(f"Repo root: {REPO_ROOT}")
print(f"Checked out commit: {head}")


Refreshing existing repo at /content/assignment2-systems
Repo root: /content/assignment2-systems
Checked out commit: 2be7db5


In [12]:
%cd /content/assignment2-systems
%pip uninstall -y -q torchaudio torchvision torchtext fastai timm
%pip install -q -e ./cs336-basics -e .


/content
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cs336_basics (pyproject.toml) ... done
  Building editable for cs336-systems (pyproject.toml) ... done


In [13]:
import os
import shutil

import psutil
import torch

def cuda_runtime_usable() -> tuple[bool, str]:
    if not torch.cuda.is_available():
        return False, "CUDA is not available in this runtime."
    try:
        _ = torch.tensor([1], device="cuda") + 1
        torch.cuda.synchronize()
        name = torch.cuda.get_device_name(0)
        capability = torch.cuda.get_device_capability(0)
        return True, f"Using CUDA on {name} with capability {capability}."
    except Exception as exc:
        return False, f"CUDA is present but unusable with this Torch build: {exc}"

cuda_ok, cuda_message = cuda_runtime_usable()
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA usable:", cuda_ok)
print("CUDA status:", cuda_message)
print("COLAB_RELEASE_TAG:", os.environ.get("COLAB_RELEASE_TAG"))
print("COLAB_GPU:", os.environ.get("COLAB_GPU"))
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU capability:", torch.cuda.get_device_capability(0))
vm = psutil.virtual_memory()
disk = shutil.disk_usage("/")
print("RAM total (GB):", round(vm.total / 1e9, 2))
print("Disk free (GB):", round(disk.free / 1e9, 2))


Torch version: 2.6.0+cu124
CUDA available: True
CUDA usable: True
CUDA status: Using CUDA on NVIDIA H100 80GB HBM3 with capability (9, 0).
COLAB_RELEASE_TAG: release-colab-external-images_20260324-060036_RC00
COLAB_GPU: 1
GPU: NVIDIA H100 80GB HBM3
GPU capability: (9, 0)
RAM total (GB): 247.01
Disk free (GB): 197.6


In [17]:
import argparse
import gc
import importlib
import importlib.util
import sys
from pathlib import Path
from timeit import default_timer

import pandas as pd
import torch
import torch.nn as nn

repo_root = Path("/content/assignment2-systems")
basics_root = repo_root / "cs336-basics"
systems_root = repo_root / "cs336_systems"
implementations_root = systems_root / "implementations"

for path in (repo_root, basics_root, systems_root, implementations_root):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

importlib.invalidate_caches()

benchmark_candidates = [
    implementations_root / "benchmarking.py",
    systems_root / "benchmarking.py",
    basics_root / "scripts" / "benchmarking.py",
]
benchmark_path = next((path for path in benchmark_candidates if path.exists()), None)
if benchmark_path is None:
    raise FileNotFoundError(f"Could not find benchmarking.py in any expected location: {benchmark_candidates}")
spec = importlib.util.spec_from_file_location("assignment2_benchmarking", benchmark_path)
benchmarking = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = benchmarking
assert spec.loader is not None
spec.loader.exec_module(benchmarking)

from cs336_basics.model import scaled_dot_product_attention

def detect_device(preferred: str = "auto") -> str:
    if preferred != "auto":
        return preferred
    if cuda_ok:
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

def cleanup_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def make_benchmark_config(
    *,
    model_size: str,
    context_length: int,
    batch_size: int,
    vocab_size: int,
    rope_theta: float,
    warmup_steps: int,
    measurement_steps: int,
    mode: str,
    preferred_device: str,
    dtype: str,
    precision_autocast: str,
    seed: int,
    optimizer_lr: float = 1e-3,
):
    args = argparse.Namespace(
        model_size=model_size,
        context_length=context_length,
        batch_size=batch_size,
        vocab_size=vocab_size,
        rope_theta=rope_theta,
        warmup_steps=warmup_steps,
        measurement_steps=measurement_steps,
        mode=mode,
        device=detect_device(preferred_device),
        dtype=dtype,
        precision_autocast=precision_autocast,
        optimizer_lr=optimizer_lr,
        memory_snapshot_path=None,
        report_peak_memory=False,
        seed=seed,
    )
    return benchmarking.resolve_config(args)

class AttentionModule(nn.Module):
    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
        return scaled_dot_product_attention(Q=q, K=k, V=v)

if not hasattr(benchmarking, "safe_benchmark_attention_module"):
    def _local_benchmark_attention_module(
        module: nn.Module,
        *,
        batch_size: int,
        seq_len: int,
        d_model: int,
        device: str,
        dtype: torch.dtype,
        warmup_steps: int,
        measurement_steps: int,
    ) -> dict[str, float]:
        def make_inputs(*, requires_grad: bool) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
            q = torch.randn(batch_size, seq_len, d_model, device=device, dtype=dtype, requires_grad=requires_grad)
            k = torch.randn(batch_size, seq_len, d_model, device=device, dtype=dtype, requires_grad=requires_grad)
            v = torch.randn(batch_size, seq_len, d_model, device=device, dtype=dtype, requires_grad=requires_grad)
            return q, k, v

        module = module.to(device)
        module.eval()

        q, k, v = make_inputs(requires_grad=False)
        for _ in range(warmup_steps):
            with torch.no_grad():
                _ = module(q, k, v)
            benchmarking.synchronize(device)

        forward_times = []
        for _ in range(measurement_steps):
            benchmarking.synchronize(device)
            start = default_timer()
            with torch.no_grad():
                _ = module(q, k, v)
            benchmarking.synchronize(device)
            forward_times.append(default_timer() - start)

        torch.cuda.reset_peak_memory_stats()
        q, k, v = make_inputs(requires_grad=True)
        out = module(q, k, v)
        benchmarking.synchronize(device)
        memory_before_backward_mb = torch.cuda.memory_allocated() / (1024**2)
        out.backward(torch.randn_like(out))
        benchmarking.synchronize(device)

        for _ in range(warmup_steps):
            q, k, v = make_inputs(requires_grad=True)
            out = module(q, k, v)
            out.backward(torch.randn_like(out))
            benchmarking.synchronize(device)

        backward_times = []
        for _ in range(measurement_steps):
            q, k, v = make_inputs(requires_grad=True)
            out = module(q, k, v)
            grad = torch.randn_like(out)
            benchmarking.synchronize(device)
            start = default_timer()
            out.backward(grad)
            benchmarking.synchronize(device)
            backward_times.append(default_timer() - start)

        forward_summary = benchmarking.summarize_timings(forward_times)
        backward_summary = benchmarking.summarize_timings(backward_times)
        return {
            "forward_mean_ms": 1000 * forward_summary["mean"],
            "forward_stdev_ms": 1000 * forward_summary["stdev"],
            "backward_mean_ms": 1000 * backward_summary["mean"],
            "backward_stdev_ms": 1000 * backward_summary["stdev"],
            "memory_before_backward_mb": memory_before_backward_mb,
        }

    def _local_safe_benchmark_attention_module(module: nn.Module, **kwargs) -> dict[str, float | str | None]:
        try:
            return {"status": "ok", **_local_benchmark_attention_module(module, **kwargs)}
        except RuntimeError as exc:
            message = str(exc)
            if "out of memory" in message.lower() or "no kernel image" in message.lower():
                return {
                    "status": f"failed: {message}",
                    "forward_mean_ms": None,
                    "forward_stdev_ms": None,
                    "backward_mean_ms": None,
                    "backward_stdev_ms": None,
                    "memory_before_backward_mb": None,
                }
            raise

    benchmarking.safe_benchmark_attention_module = _local_safe_benchmark_attention_module

if not hasattr(benchmarking, "benchmark_model_variant"):
    def _local_benchmark_model_variant(config, *, compile_model: bool) -> dict[str, float]:
        torch.manual_seed(config.seed)
        model = benchmarking.build_model(config)
        optimizer = benchmarking.build_optimizer(model, config) if hasattr(benchmarking, "build_optimizer") else None
        if compile_model:
            model = torch.compile(model)
        batch = benchmarking.make_batch(config)
        step_times = benchmarking.benchmark_steps(model, batch, config, optimizer)
        return benchmarking.summarize_timings(step_times)

    benchmarking.benchmark_model_variant = _local_benchmark_model_variant

print("Loaded benchmark module from:", benchmark_path)


Loaded benchmark module from: /content/assignment2-systems/cs336-basics/scripts/benchmarking.py


## Problem `pytorch_attention`

This section reuses the timing helpers from `benchmarking.py` and runs the required Cartesian product over batch size 8, no head dimension, `d_model in [16, 32, 64, 128]`, and `sequence_length in [256, 1024, 4096, 8192, 16384]`.


In [15]:
ATTN_DEVICE = detect_device("auto")
ATTN_BATCH_SIZE = 8
ATTN_DTYPE = torch.float32
ATTN_HEAD_DIMS = [16, 32, 64, 128]
ATTN_SEQUENCE_LENGTHS = [256, 1024, 4096, 8192, 16384]
ATTN_WARMUP_STEPS = 10
ATTN_TIMED_STEPS = 100

if ATTN_DEVICE != "cuda":
    raise RuntimeError("These attention benchmarks are intended for a CUDA runtime.")

print("Attention device:", ATTN_DEVICE)


Attention device: cuda


In [18]:
attention_rows = []

for d_model in ATTN_HEAD_DIMS:
    for seq_len in ATTN_SEQUENCE_LENGTHS:
        cleanup_cuda()
        result = benchmarking.safe_benchmark_attention_module(
            AttentionModule(),
            batch_size=ATTN_BATCH_SIZE,
            seq_len=seq_len,
            d_model=d_model,
            device=ATTN_DEVICE,
            dtype=ATTN_DTYPE,
            warmup_steps=ATTN_WARMUP_STEPS,
            measurement_steps=ATTN_TIMED_STEPS,
        )
        attention_rows.append({
            "implementation": "pytorch",
            "batch_size": ATTN_BATCH_SIZE,
            "d_model": d_model,
            "sequence_length": seq_len,
            **result,
        })

attention_df = pd.DataFrame(attention_rows)
attention_df


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:823: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:180.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


,implementation,batch_size,d_model,sequence_length,status,forward_mean_ms,forward_stdev_ms,backward_mean_ms,backward_stdev_ms,memory_before_backward_mb
0,pytorch,8,16,256,ok,0.134174,0.014555,0.408076,0.020918,36.648438
1,pytorch,8,16,1024,ok,0.240393,0.053663,0.613020,0.010285,130.593750
2,pytorch,8,16,4096,ok,2.550530,0.004057,6.135061,0.031426,1098.375000
3,pytorch,8,16,8192,ok,9.966377,0.014399,23.962730,0.074423,4180.750000
4,pytorch,8,16,16384,failed: CUDA out of memory. Tried to allocate ...,NaN,NaN,NaN,NaN,NaN
5,pytorch,8,32,256,ok,0.140783,0.010259,0.421880,0.014519,69.273438
6,pytorch,8,32,1024,ok,0.241229,0.004178,0.616648,0.007832,133.093750
7,pytorch,8,32,4096,ok,2.632513,0.005097,6.216209,0.043573,1108.375000
8,pytorch,8,32,8192,ok,10.282288,0.011566,24.223076,0.080532,4200.750000
9,pytorch,8,32,16384,failed: CUDA out of memory. Tried to allocate ...,NaN,NaN,NaN,NaN,NaN


## Problem `torch_compile` Part (a)

This section reruns the same attention grid, but compares `torch.compile(AttentionModule())` against the vanilla module.


In [ ]:
compile_attention_rows = []

for compiled in [False, True]:
    for d_model in ATTN_HEAD_DIMS:
        for seq_len in ATTN_SEQUENCE_LENGTHS:
            cleanup_cuda()
            module = AttentionModule()
            if compiled:
                module = torch.compile(module)
            result = benchmarking.safe_benchmark_attention_module(
                module,
                batch_size=ATTN_BATCH_SIZE,
                seq_len=seq_len,
                d_model=d_model,
                device=ATTN_DEVICE,
                dtype=ATTN_DTYPE,
                warmup_steps=ATTN_WARMUP_STEPS,
                measurement_steps=ATTN_TIMED_STEPS,
            )
            compile_attention_rows.append({
                "compiled": compiled,
                "batch_size": ATTN_BATCH_SIZE,
                "d_model": d_model,
                "sequence_length": seq_len,
                **result,
            })

compile_attention_df = pd.DataFrame(compile_attention_rows)
compile_attention_df


## Problem `torch_compile` Part (b)

This section uses the existing end-to-end Transformer benchmarking helpers and compares vanilla vs compiled models for both `forward` and `train-step`.


In [ ]:
TRANSFORMER_MODEL_SIZES = ["small", "medium", "large", "xl"]
TRANSFORMER_CONTEXT_LENGTH = 128
TRANSFORMER_BATCH_SIZE = 4
TRANSFORMER_VOCAB_SIZE = 10_000
TRANSFORMER_WARMUP_STEPS = 5
TRANSFORMER_TIMED_STEPS = 10
TRANSFORMER_DTYPE = "float32"
TRANSFORMER_AUTOCAST = "none"
TRANSFORMER_SEED = 0
TRANSFORMER_DEVICE = detect_device("auto")

if TRANSFORMER_DEVICE != "cuda":
    raise RuntimeError("The Transformer compile benchmark is intended for a CUDA runtime.")

print("Transformer benchmark device:", TRANSFORMER_DEVICE)


In [ ]:
transformer_rows = []

for compiled in [False, True]:
    for mode in ["forward", "train-step"]:
        for model_size in TRANSFORMER_MODEL_SIZES:
            cleanup_cuda()
            config = make_benchmark_config(
                model_size=model_size,
                context_length=TRANSFORMER_CONTEXT_LENGTH,
                batch_size=TRANSFORMER_BATCH_SIZE,
                vocab_size=TRANSFORMER_VOCAB_SIZE,
                rope_theta=10_000.0,
                warmup_steps=TRANSFORMER_WARMUP_STEPS,
                measurement_steps=TRANSFORMER_TIMED_STEPS,
                mode=mode,
                preferred_device="cuda",
                dtype=TRANSFORMER_DTYPE,
                precision_autocast=TRANSFORMER_AUTOCAST,
                seed=TRANSFORMER_SEED,
            )
            try:
                summary = benchmarking.benchmark_model_variant(config, compile_model=compiled)
                transformer_rows.append({
                    "compiled": compiled,
                    "mode": mode,
                    "model_size": model_size,
                    "mean_ms": summary["mean"] * 1000,
                    "stdev_ms": summary["stdev"] * 1000,
                    "min_ms": summary["min"] * 1000,
                    "max_ms": summary["max"] * 1000,
                    "status": "ok",
                })
            except RuntimeError as exc:
                cleanup_cuda()
                transformer_rows.append({
                    "compiled": compiled,
                    "mode": mode,
                    "model_size": model_size,
                    "mean_ms": None,
                    "stdev_ms": None,
                    "min_ms": None,
                    "max_ms": None,
                    "status": f"failed: {exc}",
                })

transformer_df = pd.DataFrame(transformer_rows)
transformer_df
